# Clustering: finding groups without labels

Companion to `clustering.html`.

1. Clustering on plain numbers (K-means, DBSCAN, hierarchical), **no embeddings**
2. Clustering text with **embeddings** (same algorithms, new input)

Install once: `pip install scikit-learn matplotlib sentence-transformers`

## Part 1: Plain numbers
Two toy datasets: round blobs, and two crescent moons.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons

X_blobs, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=7)
X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=3)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(*X_blobs.T, s=15, c="gray"); ax[0].set_title("Blobs (unlabeled)")
ax[1].scatter(*X_moons.T, s=15, c="gray"); ax[1].set_title("Moons (unlabeled)")
plt.show()

### K-means
Pick k, place k centroids, **assign** points to the nearest one, **move** each centroid to the mean of its points, repeat.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_blobs)
print("cluster sizes:", np.bincount(km.labels_))
print("inertia:", round(km.inertia_, 1))

plt.scatter(*X_blobs.T, c=km.labels_, s=15, cmap="tab10")
plt.scatter(*km.cluster_centers_.T, c="black", marker="X", s=200)
plt.title("K-means, k=3"); plt.show()

**How to choose k?** Plot inertia for several k and look for the elbow where the curve bends.

In [ ]:
ks = range(1, 9)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_blobs).inertia_ for k in ks]
plt.plot(ks, inertias, "o-"); plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Elbow plot")
plt.show()

### DBSCAN (density based)
No k needed. Points with at least `min_samples` neighbours within `eps` are core points; dense areas become clusters; the rest is noise (label `-1`).

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.2, min_samples=5).fit(X_moons)
print("clusters found:", len(set(db.labels_) - {-1}), " noise points:", (db.labels_ == -1).sum())
plt.scatter(*X_moons.T, c=db.labels_, s=15, cmap="tab10")
plt.title("DBSCAN on moons"); plt.show()

### Hierarchical (agglomerative)
Start with every point alone, repeatedly merge the two closest clusters. `linkage` defines what "closest" means.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, link in zip(ax, ["single", "average"]):
    lab = AgglomerativeClustering(n_clusters=2, linkage=link).fit_predict(X_moons)
    a.scatter(*X_moons.T, c=lab, s=15, cmap="tab10"); a.set_title(f"linkage = {link}")
plt.show()

### Same data, three algorithms
K-means assumes round blobs, so it cuts the moons with a straight line. DBSCAN and single-linkage follow the shape.

In [ ]:
models = {
    "K-means": KMeans(n_clusters=2, n_init=10, random_state=0),
    "DBSCAN": DBSCAN(eps=0.2, min_samples=5),
    "Hierarchical (single)": AgglomerativeClustering(n_clusters=2, linkage="single"),
}
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, (name, m) in zip(ax, models.items()):
    a.scatter(*X_moons.T, c=m.fit_predict(X_moons), s=15, cmap="tab10"); a.set_title(name)
plt.show()

## Part 2: Clustering text with embeddings

The algorithms above need each item to be a few meaningful numbers. A sentence is not.
An **embedding model** turns each sentence into a vector where similar meaning means nearby vectors.
Then we use the *same* K-means as before.

In [ ]:
sentences = [
    # pets
    "My cat loves chasing the puppy", "The kitten sleeps in the sun", "A dog needs a walk every day",
    "Hamsters are small and cute pets",
    # food
    "Cook the pasta with tomato sauce", "The best pizza has fresh cheese", "A bowl of hot soup for dinner",
    "Bake a chocolate cake for the party",
    # vehicles
    "The electric car has a long battery range", "A truck carried the cargo across the country",
    "The bus was late this morning", "New engine makes the motorbike faster",
    # sport
    "The football team won the final match", "She scored a goal in the last minute",
    "Tennis players practise serves daily", "The marathon runner broke the record",
]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(sentences, normalize_embeddings=True)
print(vectors.shape, "-> 16 sentences, 384 numbers each")

Cluster with K-means. We never tell it the topics.

In [ ]:
k = 4
labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(vectors)

for c in range(k):
    print(f"--- cluster {c} ---")
    for s, l in zip(sentences, labels):
        if l == c:
            print("  ", s)

Plot it. 384 numbers cannot be drawn, so reduce to 2-D with PCA **only for the picture**. Clustering itself used all 384.

In [ ]:
from sklearn.decomposition import PCA

xy = PCA(n_components=2).fit_transform(vectors)
plt.figure(figsize=(7, 5))
plt.scatter(*xy.T, c=labels, cmap="tab10", s=80)
for (x, y), s in zip(xy, sentences):
    plt.annotate(s[:22], (x, y), fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.title("Sentence embeddings, coloured by K-means cluster"); plt.show()

### The wrong k
Too small merges topics, too large splits them. Always read the clusters.

In [ ]:
for k in (2, 6):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(vectors)
    print(f"\n===== k = {k} =====")
    for c in range(k):
        print(f"cluster {c}:", [s[:25] for s, l in zip(sentences, lab) if l == c])

### Other algorithms on embeddings
DBSCAN and hierarchical clustering accept embeddings too. With normalised vectors, use cosine distance:

In [ ]:
agg = AgglomerativeClustering(n_clusters=4, metric="cosine", linkage="average").fit_predict(vectors)
for c in range(4):
    print(f"cluster {c}:", [s[:25] for s, l in zip(sentences, agg) if l == c])

## Exercises
1. In Part 1, change `n_clusters` for blobs to 2 and 5. How do the picture and inertia change?
2. Make DBSCAN `eps` tiny, then huge, on the moons. Describe both extremes.
3. Add 8 sentences about a new topic (e.g. weather). Does `k=5` find it?
4. Cluster support tickets or product reviews of your own and name each cluster.
5. Compare K-means and Agglomerative on the sentences. Do they agree?